In [ ]:
import json
import re
import time
import urllib.parse
import difflib

import pandas as pd
import numpy as np
import torch
import requests
from tqdm import tqdm
from datasets import load_dataset
from keybert import KeyBERT
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

In [ ]:
from datasets import load_dataset
import pandas as pd
import numpy as np

dataset = load_dataset("librarian-bots/arxiv-metadata-snapshot", split="train", streaming=True)

quota_per_topic = 110
buckets = {
    "Foundation_Models": [],
    "NLP_IR": [],
    "AI_for_Science": [],
    "HCI": [],
    "Robotics": []
}

keywords = {
    "Foundation_Models": ["transformer", "llm", "multimodal", "vision-language", "lora", "peft", "pre-training"],
    "NLP_IR": ["rag", "retrieval-augmented", "information retrieval", "chain-of-thought", "text mining", "long-context"],
    "AI_for_Science": ["molecule", "protein", "alignment", "graph neural", "bioinformatics", "drug discovery", "healthcare llm"],
    "HCI": ["human-computer interaction", "hci", "user interface", "human-ai", "virtual reality", "augmented reality", "accessibility"],
    "Robotics": ["robotics", "robot", "embodied ai", "navigation", "manipulation", "cobot", "soft robotics"]
}

count = 0
for paper in dataset:

    if all(len(v) >= quota_per_topic for v in buckets.values()):
        break

    title = str(paper.get('title', '')).replace('\n', ' ')
    abstract = str(paper.get('abstract', '')).replace('\n', ' ')
    text_content = (title + " " + abstract).lower()


    for topic, kw_list in keywords.items():
        if len(buckets[topic]) < quota_per_topic:
            if any(kw in text_content for kw in kw_list):
                buckets[topic].append({
                    "title": title,
                    "abstract": abstract,
                    "authors": paper.get('authors', ''),
                    "update_date": paper.get('update_date', '2026-02-20'),
                    "category": topic,
                    "text_for_embedding": f"{title}. {abstract}"
                })
                break

    count += 1
    if count % 20000 == 0:
        status = {k: len(v) for k, v in buckets.items()}
        print(f" Scanned {count} papers... Capture progress: {status}")


all_data = []
for b in buckets.values(): all_data.extend(b)
df = pd.DataFrame(all_data)


np.random.seed(42)
df['citations'] = np.random.randint(10, 1000, size=len(df))


file_name = "xiaohongshu_full_topics.csv"
df.to_csv(file_name, index=False)

print("\n" + "="*40)
print(f" Task completed! Captured a total of {len(df)} high-quality papers.")
print(f" File saved: {file_name}")
print(df['category'].value_counts())

In [ ]:
from huggingface_hub import login

login()

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig


MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"

print(f"🚀 Switching to Mistral-7B-v0.3 on A100... (The Stable Choice)")


bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )
    print("\n" + "="*40)
    print("SUCCESS: Mistral-7B-v0.3 is ONLINE!")
    print("="*40)
except Exception as e:
    print(f"\n Load failed: {e}")

In [ ]:
import pandas as pd
import json
import re
import torch
from tqdm import tqdm


df = pd.read_csv("xiaohongshu_full_topics.csv")
final_results = []


def generate_post(row):
    prompt = f"""[INST] You are a research influencer. Analyze this paper and output ONLY a valid JSON object.
Fields:
- "display_title": A viral, catchy title with relevant emojis.
- "social_content": A 3-sentence engaging summary using clear language and emojis.
- "innovation": A technical summary of the core breakthrough in LESS THAN 25 WORDS.

Title: {row['title']}
Abstract: {row['abstract']} [/INST]"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=450,
            do_sample=True,
            temperature=0.3,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(output_ids[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return response

print(f" Map Phase: Extracting features from {len(df)} papers...")
model.eval()

for i, row in tqdm(df.iterrows(), total=len(df)):
    raw_json = generate_post(row)
    try:

        match = re.search(r'\{.*\}', raw_json, re.DOTALL)
        if match:
            data = json.loads(match.group(0))
            data.update({
                "id": f"paper-{i}",
                "likesCount": int(row.get('citations', 0)),
                "tags": [str(row.get('category', 'AI'))],
                "imageUrl": f"https://picsum.photos/400/500?random={i}"
            })
            final_results.append(data)
    except:
        continue


with open("final_posts_en.json", "w", encoding="utf-8") as f:
    json.dump(final_results, f, indent=2)

print("\n Map Phase Complete! final_posts_en.json is ready.")

In [ ]:
import pandas as pd

file_path = 'xiaohongshu_full_topics.csv'

try:

    df = pd.read_csv(file_path)

    extracted_df = df[['title', 'abstract']].dropna()


    print(f" Successfully extracted {len(extracted_df)} records!")
    print("\n--- Data Preview ---")
    print(extracted_df.head())

    output_csv = 'extracted_titles_abstracts.csv'
    extracted_df.to_csv(output_csv, index=False, encoding='utf-8-sig')


    output_json = 'extracted_data.json'
    extracted_df.to_json(output_json, orient='records', force_ascii=False, indent=2)

    print(f"\n Export completed!")
    print(f"1. CSV file saved as: {output_csv}")
    print(f"2. JSON file saved as: {output_json}")

except FileNotFoundError:
    print(" Error: File not found. Please make sure you have uploaded 'xiaohongshu_full_topics.csv' to the left file panel in Colab.")
except Exception as e:
    print(f" An error occurred: {e}")

In [ ]:
import json
from keybert import KeyBERT
from tqdm import tqdm

print(" Loading KeyBERT extraction engine...")
kw_model = KeyBERT()


with open('extracted_titles_abstracts.json', 'r', encoding='utf-8') as f:
    raw_papers = json.load(f)

print(f"⚡️ Starting to analyze hardcore keywords for {len(raw_papers)} raw abstracts...")

keyword_database = []


for i, paper in enumerate(tqdm(raw_papers)):
    abstract = paper.get('abstract', '')


    extracted = kw_model.extract_keywords(
        abstract,
        keyphrase_ngram_range=(1, 2),
        stop_words='english',
        top_n=3
    )


    dynamic_kws = [kw[0].title() for kw in extracted]


    keyword_database.append({
        "id": f"paper-{i}",
        "title": paper.get('title'),
        "dynamic_keywords": dynamic_kws
    })


output_filename = "keywords_with_keybert.json"
with open(output_filename, 'w', encoding='utf-8') as f:
    json.dump(keyword_database, f, indent=2, ensure_ascii=False)

print(f"\n Perfect! Independent extraction complete.")
print(f" Pure keyword database saved as: {output_filename}")

In [ ]:
import pandas as pd
import requests
import time
import re
import urllib.parse
import difflib
from tqdm import tqdm


file_path = 'xiaohongshu_full_topics.csv'
df = pd.read_csv(file_path)

def clean_title(title):

    return re.sub(r'["\']', '', str(title)).strip()

def is_similar(title1, title2, threshold=0.85):

    if not title1 or not title2:
        return False
    t1 = title1.lower().strip()
    t2 = title2.lower().strip()
    return difflib.SequenceMatcher(None, t1, t2).ratio() >= threshold

def search_arxiv(title):
    """Fallback to ArXiv API"""
    base_url = "http://export.arxiv.org/api/query?"

    safe_title = urllib.parse.quote(title)
    params = f"search_query=ti:\"{safe_title}\"&max_results=1"
    try:
        response = requests.get(base_url + params, timeout=10)
        if response.status_code == 200 and '<entry>' in response.text:

            title_match = re.search(r'<title>(.*?)</title>', response.text, re.DOTALL)
            if title_match:
                arxiv_title = title_match.group(1).replace('\n', ' ').strip()
                if not is_similar(title, arxiv_title):
                    return None


            id_match = re.search(r'<id>http://arxiv.org/abs/(.*?)</id>', response.text)
            if id_match:
                return f"https://arxiv.org/abs/{id_match.group(1)}"
    except:
        pass
    return None

def fetch_paper_url(title):
    cleaned = clean_title(title)


    search_url = "https://api.semanticscholar.org/graph/v1/paper/search"

    params = {"query": cleaned, "limit": 1, "fields": "title,url,openAccessPdf"}

    try:
        response = requests.get(search_url, params=params, timeout=10)
        if response.status_code == 200:
            data = response.json()
            if data.get('total', 0) > 0:
                paper = data['data'][0]
                returned_title = paper.get('title', '')


                if is_similar(cleaned, returned_title):
                    pdf_url = paper.get('openAccessPdf', {}).get('url') if paper.get('openAccessPdf') else None
                    if pdf_url: return pdf_url
                    if paper.get('url'): return paper['url']
    except:
        pass


    arxiv_url = search_arxiv(cleaned)
    if arxiv_url: return arxiv_url


    safe_query = urllib.parse.quote_plus(cleaned)
    return f"https://scholar.google.com/scholar?q={safe_query}"

# 2. Process all papers
print(f"🔍 Starting robust search for {len(df)} papers...")
results = []

for i, row in tqdm(df.iterrows(), total=len(df)):
    link = fetch_paper_url(row['title'])
    results.append(link)

    time.sleep(1.0)

df['paper_url'] = results
df.to_csv("papers_with_urls_v2_fixed.csv", index=False, encoding='utf-8-sig')
print("\n Finished! Check 'papers_with_urls_v2_fixed.csv'")